In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

In [2]:
# --- 1. Load and Inspect Data ---
# Use a raw string for the Windows file path.
data_path = r"C:\Users\ronle\Desktop\BUAS\Y2C\data_raw.csv"
df = pd.read_csv(data_path)

In [3]:
# Display the first few rows to verify the structure
print(df.head())

   Unnamed: 0                                               text  happy  sad  \
0           0                                    That game hurt.      0    1   
1           2     You do right, if you don't care then fuck 'em!      0    0   
2           3                                 Man I love reddit.      1    0   
3           4  [NAME] was nowhere near them, he was by the Fa...      0    0   
4           5  Right? Considering it’s such an important docu...      1    0   

   disgusted  mad  scared  surprised  neutral  \
0          0    0       0          0        0   
1          0    0       0          0        1   
2          0    0       0          0        0   
3          0    0       0          0        1   
4          0    0       0          0        0   

                                            POS_Tags  ... Sentiment  \
0  [('That', 'DT'), ('game', 'NN'), ('hurt', 'VBD...  ...  Negative   
1  [('You', 'PRP'), ('do', 'VBP'), ('right', 'RB'...  ...   Neutral   
2  [('Man',

In [4]:
# --- 2. Prepare the Labels ---
# Assuming the columns for emotions are named exactly as follows:
emotion_cols = ['happy', 'sad', 'disgusted', 'mad', 'scared', 'surprised', 'neutral']
# For single emotion classification, we pick the index of the column with the highest value (assumed to be 1)
labels = df[emotion_cols].values
# Get the class index for each sample
y = np.argmax(labels, axis=1)
# Convert to one-hot encoding for categorical crossentropy
y_cat = to_categorical(y, num_classes=len(emotion_cols))

In [5]:
# --- 3. Prepare the Text Data ---
texts = df['text'].astype(str).tolist()  # ensure all texts are strings

# Set parameters for tokenization and padding
max_num_words = 10000  # maximum number of words to consider in the vocabulary
max_sequence_length = 100  # adjust based on your data (or use the maximum length in your dataset)
embedding_dim = 100  # size of the embedding vector

tokenizer = Tokenizer(num_words=max_num_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index
print("Found %s unique tokens." % len(word_index))

# Pad sequences so that each input has the same length
X = pad_sequences(sequences, maxlen=max_sequence_length, padding='post')

Found 32766 unique tokens.


In [6]:
# --- 4. Split Data into Training and Validation Sets ---
X_train, X_val, y_train, y_val = train_test_split(X, y_cat, test_size=0.2, random_state=42)

In [ ]:
# --- 5. Build the LSTM Model ---
model = Sequential()
model.add(Embedding(input_dim=min(max_num_words, len(word_index) + 1),
                    output_dim=embedding_dim,
                    input_length=max_sequence_length))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.1))
model.add(Dense(len(emotion_cols), activation='softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding (Embedding)       (None, 100, 100)          1000000   
                                                                 
 lstm (LSTM)                 (None, 128)               117248    
                                                                 
 dense (Dense)               (None, 7)                 903       
                                                                 
Total params: 1,118,151
Trainable params: 1,118,151
Non-trainable params: 0
_________________________________________________________________


In [8]:
# --- 6. Train the Model ---
epochs = 5
batch_size = 32

history = model.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=epochs,
                    batch_size=batch_size)

Epoch 1/5
4296/4296 [==============================] - 422s 98ms/step - loss: 1.5012 - accuracy: 0.3737 - val_loss: 1.5040 - val_accuracy: 0.3784
Epoch 2/5
4296/4296 [==============================] - 439s 102ms/step - loss: 1.4990 - accuracy: 0.3765 - val_loss: 1.5026 - val_accuracy: 0.3784
Epoch 3/5
4296/4296 [==============================] - 449s 104ms/step - loss: 1.4988 - accuracy: 0.3768 - val_loss: 1.5024 - val_accuracy: 0.3784
Epoch 4/5
4296/4296 [==============================] - 449s 105ms/step - loss: 1.4985 - accuracy: 0.3769 - val_loss: 1.5014 - val_accuracy: 0.3784
Epoch 5/5
4296/4296 [==============================] - 447s 104ms/step - loss: 1.4986 - accuracy: 0.3770 - val_loss: 1.5020 - val_accuracy: 0.3784


In [ ]:
# --- 7. Save the Model (optional) ---
# model.save("emotion_lstm_model.h5")